In [0]:
spark.sql("""
CREATE SCHEMA IF NOT EXISTS netflix_catalog.gold
""")

In [0]:
%sql
SHOW TABLES IN netflix_catalog.gold;

In [0]:
%sql
DROP TABLE IF EXISTS netflix_catalog.gold.dim_cast;
DROP TABLE IF EXISTS netflix_catalog.gold.dim_category;
DROP TABLE IF EXISTS netflix_catalog.gold.dim_country;
DROP TABLE IF EXISTS netflix_catalog.gold.dim_date;
DROP TABLE IF EXISTS netflix_catalog.gold.dim_director;
DROP TABLE IF EXISTS netflix_catalog.gold.dim_title;
DROP TABLE IF EXISTS netflix_catalog.gold.fact_title;

In [0]:
%sql
SHOW TABLES IN netflix_catalog.gold;

In [0]:
spark.sql("""
SHOW SCHEMAS IN netflix_catalog
""").display()

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable


In [0]:
SILVER_PATH = "abfss://silver@pravdatalake.dfs.core.windows.net"
GOLD_PATH = "abfss://gold@pravdatalake.dfs.core.windows.net"

In [0]:
silver_titles = spark.read.format("delta").load(
    f"{SILVER_PATH}/netflix_titles"
)

silver_directors = spark.read.format("delta").load(
    f"{SILVER_PATH}/netflix_directors"
)

silver_countries = spark.read.format("delta").load(
    f"{SILVER_PATH}/netflix_countries"
)

silver_category = spark.read.format("delta").load(
    f"{SILVER_PATH}/netflix_category"
)

silver_cast = spark.read.format("delta").load(
    f"{SILVER_PATH}/netflix_cast"
)

In [0]:
silver_titles.count()

####Create dim_date

In [0]:
df_dim_date = (
    silver_titles
    .select(
        to_date("date_added").alias("full_date")
    )
    .filter(col("full_date").isNotNull())
    .distinct()
    .withColumn(
        "date_sk",
        date_format("full_date", "yyyyMMdd").cast("int")
    )
    .withColumn("day", dayofmonth("full_date"))
    .withColumn("month", month("full_date"))
    .withColumn("month_name", date_format("full_date", "MMMM"))
    .withColumn("quarter", quarter("full_date"))
    .withColumn("year", year("full_date"))
    .withColumn("day_of_week", dayofweek("full_date"))
    .withColumn("day_name", date_format("full_date", "EEEE"))
    .withColumn(
        "is_weekend",
        dayofweek("full_date").isin([1, 7])
    )
    .select(
        "date_sk",
        "full_date",
        "day",
        "month",
        "month_name",
        "quarter",
        "year",
        "day_of_week",
        "day_name",
        "is_weekend"
    )
)

display(df_dim_date)

In [0]:

DIM_DATE_TABLE = "netflix_catalog.gold.dim_date"

if spark.catalog.tableExists(DIM_DATE_TABLE):

    delta_tbl = DeltaTable.forPath(
        spark,
        f"{GOLD_PATH}/dim_date"
    )

    delta_tbl.alias("trg") \
        .merge(
            df_dim_date.alias("src"),
            "trg.date_sk = src.date_sk"
        ) \
        .whenNotMatchedInsertAll() \
        .execute()

else:

    df_dim_date.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{GOLD_PATH}/dim_date") \
        .saveAsTable(DIM_DATE_TABLE)

print("dim_date completed.")

In [0]:
spark.sql("""
          SELECT * FROM netflix_catalog.gold.dim_date
          """).display()

####Create dim_country

In [0]:
df_dim_country = (
        silver_countries
        .select("country")
        .distinct()
        .withColumn(
            "country_sk",
            xxhash64("country")
        )
        .select(
            "country_sk",
            "country"
        )
    )

display(df_dim_country)

In [0]:

DIM_COUNTRY_TABLE = "netflix_catalog.gold.dim_country"

if spark.catalog.tableExists(DIM_COUNTRY_TABLE):

    delta_tbl = DeltaTable.forPath(
        spark,
        f"{GOLD_PATH}/dim_country"
    )

    delta_tbl.alias("trg") \
        .merge(
            df_dim_country.alias("src"),
            "trg.country_sk = src.country_sk"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

else:

    df_dim_country.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{GOLD_PATH}/dim_country") \
        .saveAsTable(DIM_COUNTRY_TABLE)

print("dim_country completed - SCD Type 1.")

In [0]:
spark.sql("""
          SELECT * FROM netflix_catalog.gold.dim_country
          """).display()

####Create dim_category

In [0]:
df_dim_category = (
    silver_category
    .select("listed_in")
    .filter(col("listed_in").isNotNull())
    .withColumn("category", explode(split(col("listed_in"), ",")))
    .withColumn("category", trim(col("category")))
    .select("category")
    .distinct()
)

In [0]:
DIM_CATEGORY_TABLE = "netflix_catalog.gold.dim_category"

df_dim_category.write \
    .format("delta") \
    .mode("overwrite") \
    .option("path", f"{GOLD_PATH}/dim_category") \
    .saveAsTable(DIM_CATEGORY_TABLE)

print("dim_category created successfully")

In [0]:
spark.sql("""
          SELECT * FROM netflix_catalog.gold.dim_category
          """).display()

####Create dim_director

In [0]:
df_dim_director = (
    silver_directors
    .select("director")
    .withColumn(
        "director_sk",
        xxhash64("director")
    )
    .select(
        "director_sk",
        "director"
    )
)

display(df_dim_director)

In [0]:

DIM_DIRECTOR_TABLE = "netflix_catalog.gold.dim_director"

if spark.catalog.tableExists(DIM_DIRECTOR_TABLE):

    delta_tbl = DeltaTable.forPath(
        spark,
        f"{GOLD_PATH}/dim_directors"
    )

    delta_tbl.alias("trg") \
        .merge(
            df_dim_director.alias("src"),
            "trg.director_sk = src.director_sk"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

else:

    df_dim_director.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{GOLD_PATH}/dim_directors") \
        .saveAsTable(DIM_DIRECTOR_TABLE)

print("dim_director completed - SCD Type 1.")

In [0]:
spark.sql("""
          SELECT * FROM netflix_catalog.gold.dim_director
          """).display()

####Create dim_cast

In [0]:
df_dim_cast = (
    silver_cast
    .select("cast")
    .withColumn(
        "cast_sk",
        xxhash64("cast")
    )
    .select(
        "cast_sk",
        col("cast").alias("cast_member")
    )
)

display(df_dim_cast)

In [0]:

DIM_CAST_TABLE = "netflix_catalog.gold.dim_cast"

if spark.catalog.tableExists(DIM_CAST_TABLE):

    delta_tbl = DeltaTable.forPath(
        spark,
        f"{GOLD_PATH}/dim_cast"
    )

    delta_tbl.alias("trg") \
        .merge(
            df_dim_cast.alias("src"),
            "trg.cast_sk = src.cast_sk"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

else:

    df_dim_cast.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{GOLD_PATH}/dim_cast") \
        .saveAsTable(DIM_CAST_TABLE)

print("dim_cast completed - SCD Type 1.")

In [0]:
spark.sql("""
          SELECT * FROM netflix_catalog.gold.dim_cast
          """).display()

####Create dim_title

In [0]:
df_dim_title = (
    silver_titles
    .select(
        "show_id",
        "type",
        "title",
        "date_added",
        "release_year",
        "rating",
        "description",
        "duration_minutes",
        "duration_seasons"
    )
)


In [0]:
DIM_TITLE_TABLE = "netflix_catalog.gold.dim_title"

if not spark.catalog.tableExists(DIM_TITLE_TABLE):

    df_initial = (
        df_dim_title
        .withColumn("effective_start_date", current_date())
        .withColumn(
            "effective_end_date",
            to_date(lit("9999-12-31"))
        )
        .withColumn("is_current", lit(True))
    )

    df_initial.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{GOLD_PATH}/dim_title") \
        .saveAsTable(DIM_TITLE_TABLE)

    print("dim_title created successfully.")

In [0]:
current_titles = (
    spark.table(DIM_TITLE_TABLE)
    .filter(col("is_current") == True)
)

In [0]:
changed_titles = (
    df_dim_title.alias("src")
    .join(
        current_titles.alias("trg"),
        "show_id",
        "left"
    )
    .filter(
        col("trg.show_id").isNull()
        |
        (
            (col("src.type") != col("trg.type"))
            |
            (col("src.title") != col("trg.title"))
            |
            (col("src.date_added") != col("trg.date_added"))
            |
            (col("src.release_year") != col("trg.release_year"))
            |
            (col("src.rating") != col("trg.rating"))
            |
            (col("src.description") != col("trg.description"))
            |
            (col("src.duration_minutes") != col("trg.duration_minutes"))
            |
            (col("src.duration_seasons") != col("trg.duration_seasons"))
        )
    )
    .select("src.*")
)

In [0]:
delta_tbl = DeltaTable.forPath(
    spark,
    f"{GOLD_PATH}/dim_title"
)

delta_tbl.alias("trg") \
    .merge(
        changed_titles.alias("src"),
        """
        trg.show_id = src.show_id
        AND trg.is_current = true
        """
    ) \
    .whenMatchedUpdate(
        set={
            "effective_end_date": date_sub(current_date(), 1),
            "is_current": lit(False)
        }
    ) \
    .execute()

In [0]:
new_versions = (
    changed_titles
    .withColumn(
        "effective_start_date",
        current_date()
    )
    .withColumn(
        "effective_end_date",
        to_date(lit("9999-12-31"))
    )
    .withColumn(
        "is_current",
        lit(True)
    )
)

new_versions.write \
    .format("delta") \
    .mode("append") \
    .save(f"{GOLD_PATH}/dim_title")

print("dim_title SCD Type 2 completed.")

In [0]:
display(
    spark.table(DIM_TITLE_TABLE)
    .orderBy("show_id", "effective_start_date")
)

####Create fact_title

In [0]:

df_fact_title = (
    silver_titles
    .select(
        "show_id",
        "date_added",
        "release_year",
        "duration_minutes",
        "duration_seasons"
    )
    .withColumn(
        "date_sk",
        date_format(
            to_date("date_added"),
            "yyyyMMdd"
        ).cast("int")
    )
    .withColumn(
        "title_count",
        lit(1)
    )
)

In [0]:
FACT_TITLE_TABLE = "netflix_catalog.gold.fact_title"

if spark.catalog.tableExists(FACT_TITLE_TABLE):

    delta_tbl = DeltaTable.forPath(
        spark,
        f"{GOLD_PATH}/fact_title"
    )

    delta_tbl.alias("trg") \
        .merge(
            df_fact_title.alias("src"),
            "trg.show_id = src.show_id"
        ) \
        .whenMatchedUpdateAll() \
        .whenNotMatchedInsertAll() \
        .execute()

else:

    df_fact_title.write \
        .format("delta") \
        .mode("overwrite") \
        .option("path", f"{GOLD_PATH}/fact_title") \
        .saveAsTable(FACT_TITLE_TABLE)

print("fact_title completed.")

In [0]:
spark.sql("""
SHOW TABLES IN netflix_catalog.gold
""").show(truncate=False)

In [0]:
tables = [
    "dim_date",
    "dim_country",
    "dim_category",
    "dim_director",
    "dim_cast",
    "dim_title",
    "fact_title"
]

for table in tables:
    count = spark.table(
        f"netflix_catalog.gold.{table}"
    ).count()

    print(f"{table}: {count}")